<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2025 Google LLC。

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/core/keras_inference"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td>    <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/docs/core/keras_inference.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/keras_inference.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fdocs%2Fcore%2Fkeras_inference.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/docs/core/keras_inference.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

# 使用Keras執行Gemma

產生內容、總結和分析內容只是您可以使用 Gemma 開放式模型完成的部分任務。本教學向您展示如何開始使用Keras執行Gemma，包括使用文字和圖像輸入生成文字內容。 [Keras](https://keras.io/) 提供使用 JAX、PyTorch 和 TensorFlow 執行 Gemma 和其他模型的實作。如果您是 Keras 的新手，您可能需要在開始之前閱讀 [Keras 入門](https://keras.io/getting_started/)。
Gemma 3以上型號支援文字和圖像輸入。 Gemma 的早期版本僅支援文字輸入，某些變體除外，包括 [PaliGemma](https://ai.google.dev/gemma/docs/setup)。

## 設定

在開始本教學之前，請確保您已完成以下步驟：
* 在 [kaggle.com](https://www.kaggle.com) 上造訪Gemma。
* 選擇具有足夠資源執行的 Colab runtime
您要執行的 Gemma 模型大小。 [了解更多](https://ai.google.dev/gemma/docs/core#sizes)。* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

如果您需要協助完成這些步驟，請參閱[Gemma 設定](https://ai.google.dev/gemma/docs/setup) 說明。完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝 Keras 軟體包

安裝 Keras 和 KerasHub Python 軟體包。

In [ ]:
!pip install -q -U keras-hub
!pip install -q -U keras

### 選擇後端

Keras 是高級、多framework 深度學習API，設計簡單易用。 [Keras 3](https://keras.io/keras_3) 讓您選擇後端：TensorFlow、JAX 或 PyTorch。這三個都適用於本教學。在本教學中，為 JAX 設定後端，因為它通常可以提供更好的效能。

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "tensorflow" or "torch".
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

### 導入包

導入 Keras 和 KerasHub 套件。

In [ ]:
import keras
import keras_hub

## 負載模型

Keras 提供了許多流行的[模型架構](https://keras.io/api/keras_nlp/models/) 的實作。使用 `Gemma3CausalLM` 類別下載並設定 Gemma 模型，為 Gemma 3 模型建立端到端的因果語言建模實作。使用`from_preset()`方法建立模型，如下列程式碼範例所示：

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset(
    "gemma3_instruct_4b",
    dtype="bfloat16",
)

`Gemma3CausalLM.from_preset()` 方法根據預設的架構和權重實例化模型。在上面的程式碼中，字串`"gemma#_xxxxxxx"`指定Gemma的預設版本和參數大小。您可以在 [Kaggle](https://www.kaggle.com/models/keras/gemma3) 上的 **型號變體** 清單中找到 Gemma 型號的程式碼字串。

下載模型後，使用 `summary()` 函數獲取有關模型的更多資訊：

In [ ]:
gemma_lm.summary()

Model: "gemma3_causal_lm_1"

摘要的輸出顯示了模型可訓練參數的總數。
為了命名模型，嵌入層不計入參數數量。

注意：要使用 Google Colab 執行更大的 Gemma 模型，您需要存取付費計劃中提供的高級 GPU。或者，您可以使用 [Kaggle](https://www.kaggle.com/code) notebook 或 Google Cloud 項目執行inference。

## 用文本生成文本

使用您在前面步驟中設定的 Gemma 模型物件的 `generate()` 方法產生帶有文字 prompt 的文字。可選的`max_length`參數指定生成序列的最大長度。以下程式碼範例顯示了 prompt 模型的幾種方法。

In [ ]:
gemma_lm.generate("what is keras in 3 bullet points?", max_length=64)

您也可以使用清單作為輸入來提供批次 prompts：

In [ ]:
gemma_lm.generate(
    ["what is keras in 3 bullet points?",
     "The universe is"],
    max_length=64)

如果您在 JAX 或 TensorFlow 後端上執行，您應該注意到第二個 `generate()` 呼叫會更快地返回答案。這種效能改進是因為對於給定批次大小和 `max_length` 的每次呼叫都是使用 XLA 進行編譯的。第一次執行成本較高，但後續執行速度更快。

### 使用 prompt 模板

當建立更複雜的請求或多輪聊天互動時，請使用 prompt 模板來建立您的請求。以下程式碼為 Gemma prompts 建立標準範本：

In [1]:
PROMPT_TEMPLATE = """<start_of_turn>user
{question}
<end_of_turn>
<start_of_turn>model
"""

以下程式碼展示如何使用範本來格式化簡單的請求：

In [ ]:
question = """"what is keras in 3 bullet points?"""
prompt = PROMPT_TEMPLATE.format(question=question)
gemma_lm.generate(prompt)

### 可選：嘗試不同的採樣器

您可以透過在`compile()`上設定`sampler`參數來控制模型物件的生成策略。預設情況下，將使用 [`"greedy"`](https://keras.io/api/keras_nlp/samplers/greedy_sampler/#greedysampler-class) 取樣。作為實驗，嘗試設定 [`"top_k"`](https://keras.io/api/keras_nlp/samplers/top_k_sampler/) 策略：

In [ ]:
gemma_lm.compile(sampler="top_k")
gemma_lm.generate("The universe is", max_length=64)

預設的貪婪演算法總是選擇機率最大的token，而 top-K 演算法則從 top K 機率的tokens中隨機選擇下一個token。您不必指定採樣器，並且如果最後一個程式碼片段對您的用例沒有幫助，您可以忽略它。如果您想了解有關可用採樣器的更多信息，請參閱[採樣器](https://keras.io/api/keras_nlp/samplers/)。

## 用圖像資料生成文字

對於Gemma 3 及更高版本的型號，您可以使用圖像作為prompt 的一部分來產生輸出。此功能可讓您使用 Gemma 來解釋視覺內容或使用圖像作為內容生成的資料。

### 建立圖像載入函數

以下函數從 URL 載入圖像文件，並將其 token 化以在 Gemma prompt 中使用：

In [ ]:
import numpy as np
import PIL

def read_image(url):
    """Reads image from URL as NumPy array."""

    image_path = keras.utils.get_file(origin=url)
    image = PIL.Image.open(image_path)
    image = np.array(image)
    return image

### 載入 prompt 的圖片

載入圖像並格式化數據，以便模型可以處理它。使用上一節中定義的`read_image()`函數，如下範例程式碼所示：

In [ ]:
from matplotlib import pyplot as plt

image = read_image(
    "https://ai.google.dev/gemma/docs/images/thali-indian-plate.jpg"
)
plt.imshow(image)

<img src="/gemma/docs/images/thali-indian-plate.jpg" />
**圖 1.** 金屬板上的塔利印度食物圖像。

### 使用映像執行請求

當prompt使用圖像內容Gemma模型時，您可以在prompt中使用特定的字串序列`<start_of_image>`將圖像作為prompt的一部分包含在內。使用 prompt 範本（例如先前定義的 `PROMPT_TEMPLATE` 字串）來格式化您的請求，如下 prompt 程式碼所示：

In [ ]:
question = """Which cuisine is this: <start_of_image>? \
Identify the food items present. Which macros is the meal \
high and low on? Keep your answer short.\
"""

gemma_lm.generate(
    {
        "images": image,
        "prompts": PROMPT_TEMPLATE.format(question=question),
    },
)

如果您使用較小的 GPU，並遇到記憶體不足 (OOM) 錯誤，則可以將 `max_images_per_prompt` 和 `sequence_length` 設定為較小的值。以下程式碼顯示如何將序列長度減少到 768。

In [ ]:
gemma_lm.preprocessor.max_images_per_prompt = 2
gemma_lm.preprocessor.sequence_length = 768

### 使用多個映像執行請求

當在 prompt 中使用多個圖像時，請為每個提供的圖像使用多個 `<start_of_image>` tokens，如下例所示：

In [ ]:
dog_a = read_image("http://localhost/images/dog-a.jpg")
dog_b = read_image("http://localhost/images/dog-b.jpg")

question = """I have two images:

Dog A: <start_of_image>
Dog B: <start_of_image>

Which breeds are they? Tell me a bit about them. \
Keep it short.\
"""

gemma_lm.generate(
    {
        "images": [dog_a, dog_b],
        "prompts": PROMPT_TEMPLATE.format(question=question),
    },
)

## 接下來是什麼

在本教學中，您學習如何使用Keras 和Gemma 生成文字。以下是關於下一步學習內容的一些建議：
* 了解如何[微調 Gemma 型號](https://ai.google.dev/gemma/docs/core/lora_tuning)。
* 了解如何[在Gemma 模型上執行分佈式fine-tuning 和inference](https://ai.google.dev/gemma/docs/core/distributed_tuning)。
* 了解 [Gemma 與 Vertex AI 整合](https://ai.google.dev/gemma/docs/integrations/vertex)
* 了解如何[將 Gemma 模型與 Vertex AI 一起使用](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。